<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L54_Evaluation_Frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L54: Evaluation Frameworks - Benchmarks and Automated Testing

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 54 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Run a small benchmark (e.g. MMLU subset or custom QA) and compute accuracy
- Combine multiple metrics (BLEU, ROUGE, F1, accuracy) in one pipeline
- Structure an evaluation config (dataset, metrics, prompts) for reproducibility

---

## Concept: Evaluation Frameworks

**Benchmarks**: MMLU, HELM, HumanEval, etc. **Metrics**: task-specific (accuracy, F1, BLEU, ROUGE, pass@k). **Pipeline**: load model → for each example: prompt + generate → score with metric → aggregate. We build a minimal pipeline with multiple metrics.

---

In [ ]:
!pip install transformers torch evaluate -q
from transformers import pipeline
from evaluate import load
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Custom Eval Dataset and Generation

---

In [ ]:
eval_data = [
    {"prompt": "2+2=", "reference": "4", "task": "math"},
    {"prompt": "The capital of France is", "reference": "Paris", "task": "qa"},
]

generator = pipeline("text-generation", model="gpt2")
generator.tokenizer.pad_token = generator.tokenizer.eos_token

def get_prediction(prompt, max_new=10):
    out = generator(prompt, max_new_tokens=max_new, do_sample=False, pad_token_id=generator.tokenizer.eos_token_id)
    return out[0]["generated_text"][len(prompt):].strip().split()[0] if out else ""

predictions = [get_prediction(d["prompt"]) for d in eval_data]
references = [d["reference"] for d in eval_data]
print("Predictions:", predictions)

## Part 2: Multiple Metrics

---

In [ ]:
bleu = load("bleu")
rouge = load("rouge")

refs_bleu = [[r] for r in references]
bleu_score = bleu.compute(predictions=predictions, references=refs_bleu)
rouge_score = rouge.compute(predictions=predictions, references=references)

accuracy = sum(1 for p, r in zip(predictions, references) if r.lower() in p.lower()) / len(predictions)
print("BLEU:", bleu_score["bleu"])
print("ROUGE-1:", rouge_score["rouge1"])
print("Match accuracy:", accuracy)

## Part 3: Eval Config (Concept)

---

In [ ]:
# Use a YAML/JSON config: dataset path, prompt template, metrics list, model name. Run script to produce results JSON.
print("Frameworks: lm-eval, HELM, EleutherAI harness. Define configs for reproducibility.")

## Exercises

1. Add 10 more QA pairs and report accuracy and BLEU.
2. Define a prompt template (e.g. "Question: {} Answer:") and compare two models.
3. Run lm-evaluation-harness on one task (e.g. hellaswag) and record score.

---

## Key Takeaways

1. Eval pipeline = dataset + model + prompt → predictions → metrics → report.
2. Use multiple metrics (accuracy, BLEU, ROUGE) for different tasks.
3. Config-driven evals improve reproducibility and comparison.

---

## Next Lesson

**L55: Prompt Injection and Security** – Attacks and defenses.

---